# 01 Book formats

**Graduates to:** `hub/src/hudhud_hub/library.py`

When you drop a file on Hudhud, the hub has to answer five questions before
the book can enter the library:

1. **What is it?** Never trust the extension. Look at the first bytes (the
   "magic number"): PDFs start with `%PDF-`, EPUB and CBZ are ZIP archives
   (`PK\x03\x04`), MOBI and AZW3 are Palm databases with `BOOKMOBI` at byte 60,
   FB2 is XML with a `<FictionBook>` root, and plain text is whatever is left
   if it decodes as UTF-8.
2. **Can we read it?** DRM-protected books are rejected with a clear reason.
   - PDF: a PDF that *needs a password to open* is rejected. A PDF encrypted
     only with an owner password (it restricts printing, but anyone can open
     it) is fine.
   - EPUB: `META-INF/rights.xml` means Adobe DRM. `META-INF/encryption.xml` is
     DRM unless every algorithm in it is font obfuscation
     (`http://www.idpf.org/2008/embedding` or `http://ns.adobe.com/pdf/enc#RC`),
     which many ordinary EPUBs use.
   - MOBI/AZW3: the PalmDOC header in record 0 has an `encryption` field at
     byte 12; anything other than 0 is DRM.
3. **What is its identity?** The sha256 of the file bytes. The same file
   imported twice gets the same id, so import is idempotent, and the stored
   file is named by that hash, never by a name the user typed.
4. **What is it called?** Title, author and language come from the format's
   own metadata. When a format has none (TXT, CBZ), use the original filename.
5. **What does it look like?** A cover image: page 1 of a PDF, the cover item
   of an EPUB, the first page of a comic. Other formats get no cover yet; the
   client draws a placeholder.

### The MOBI layout in one picture

```
file offset 0   PDB header (78 bytes)      name, ... , "BOOK" "MOBI" at 60..68
            76  uint16 number of records
            78  record list, 8 bytes each: uint32 offset, attributes, uid
record 0    +0  PalmDOC header (16 bytes)  +12 uint16 encryption
            +16 "MOBI" header              +20 uint32 header length
                                           +28 uint32 text encoding (65001 = UTF-8)
                                           +36 uint32 file version (8 = KF8/AZW3)
                                           +84 uint32 full-name offset, +88 length
                                           +128 uint32 EXTH flags (0x40 = has EXTH)
            +16+header length  "EXTH", uint32 length, uint32 count,
                               then records: uint32 type, uint32 length, data
                               (100 = author, 503 = title, 524 = language)
```

All integers are big-endian: `struct.unpack_from(">I", data, offset)`.

### Failure is a value

Every way an import can fail becomes `ImportRejected(reason)`. A broken ZIP,
truncated MOBI or malformed XML raises its own exception type deep inside a
parser; `import_book` converts those to `ImportRejected("corrupt file")`, so
the API layer only has to handle one exception. And nothing is written to the
library until every check has passed.

### Archives can lie

A ZIP is compressed, so a 300 KB upload can hold a 30 GB file (a "zip bomb").
Before reading any entry, check its declared size (`z.getinfo(name).file_size`)
against `MAX_MEMBER_BYTES`. Python's zipfile never returns more than the
declared size, so the check really bounds memory. Broken entries can also
raise `zlib.error` (bad compressed data) or `RuntimeError` (a password-
protected entry); those are corrupt files too.

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-or-later
import codecs
import hashlib
import os
import re
import shutil
import struct
import urllib.parse
import xml.etree.ElementTree as ET
import zipfile
import zlib
from dataclasses import dataclass
from pathlib import Path, PurePosixPath

import ipytest
import pymupdf

ipytest.autoconfig(raise_on_error=os.environ.get("HUDHUD_NB_CHECK") == "1")

MAX_BYTES = 500 * 1024 * 1024
MAX_MEMBER_BYTES = 50 * 1024 * 1024  # largest single file we will decompress
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".gif", ".webp"}
FONT_OBFUSCATION = {"http://www.idpf.org/2008/embedding", "http://ns.adobe.com/pdf/enc#RC"}
NS = {
    "c": "urn:oasis:names:tc:opendocument:xmlns:container",
    "opf": "http://www.idpf.org/2007/opf",
    "dc": "http://purl.org/dc/elements/1.1/",
}
FB2 = {"fb": "http://www.gribuser.ru/xml/fictionbook/2.0"}


class ImportRejected(Exception):
    """The file cannot enter the library. `reason` is shown to the user."""

    def __init__(self, reason: str):
        super().__init__(reason)
        self.reason = reason


@dataclass(frozen=True)
class Metadata:
    title: str | None
    author: str | None
    language: str | None


@dataclass(frozen=True)
class BookRecord:
    id: str
    title: str
    author: str | None
    language: str | None
    format: str
    file_size: int
    file_path: str
    cover_path: str | None

## Identity and format

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Hex sha256 of the file, read in chunks so a 500 MB PDF never sits in memory."""
    raise NotImplementedError


def detect_format(path: Path) -> str:
    """Return pdf, epub, mobi, azw3, fb2, cbz or txt from the file's first bytes.

    Read only the first 4096 bytes. Raise ImportRejected("unsupported format")
    when nothing matches, and ImportRejected("corrupt file") for a broken ZIP.
    """
    raise NotImplementedError

## Format helpers

Small functions, one job each. `detect_format` and `import_book` call them.

In [ ]:
def _looks_like_text(path: Path) -> bool:
    """True if the first 64 KiB contain no NUL byte and decode as UTF-8.

    Use codecs.getincrementaldecoder("utf-8")().decode(chunk, final=False) so a
    multi-byte character cut at the chunk edge is not an error.
    """
    raise NotImplementedError


def _read_member(z: zipfile.ZipFile, name: str) -> bytes:
    """Read one archive entry, but raise ImportRejected("archive entry too large")
    if its declared size exceeds MAX_MEMBER_BYTES."""
    raise NotImplementedError


def _zip_kind(path: Path) -> str:
    """'epub' if the `mimetype` entry is application/epub+zip OR the archive has
    META-INF/container.xml (some EPUBs omit mimetype, and the DRM check must
    still run). 'cbz' only if every file is an image (ComicInfo.xml allowed),
    so a .docx full of pictures is not a comic."""
    raise NotImplementedError


def _mobi_record0(path: Path) -> bytes:
    """Read the record list, seek to record 0 and return up to 64 KiB of it.

    Raise ImportRejected("corrupt file") if the file is too short, has no
    records, or record 0 lacks the "MOBI" identifier at offset 16.
    """
    raise NotImplementedError


def _mobi_version(path: Path) -> int:
    """The MOBI file version (uint32 at record 0 + 36)."""
    raise NotImplementedError


def _mobi_encrypted(rec0: bytes) -> bool:
    """True if the PalmDOC encryption field (uint16 at +12) is not zero."""
    raise NotImplementedError


def _mobi_metadata(rec0: bytes) -> Metadata:
    """Title from EXTH 503 (else the full name), author from EXTH 100, language from 524."""
    raise NotImplementedError


def _epub_drm(z: zipfile.ZipFile) -> bool:
    """rights.xml, or encryption.xml with any non-font-obfuscation algorithm."""
    raise NotImplementedError


def _epub_opf(z: zipfile.ZipFile) -> tuple[str, ET.Element]:
    """Follow META-INF/container.xml to the OPF; return (opf path, parsed root)."""
    raise NotImplementedError


def _epub_metadata(opf: ET.Element) -> Metadata:
    """dc:title, dc:creator and dc:language from the OPF metadata block."""
    raise NotImplementedError


def _epub_cover(z: zipfile.ZipFile, opf_path: str, opf: ET.Element) -> bytes | None:
    """Cover bytes: the manifest item with properties="cover-image" (EPUB 3), else
    the item named by <meta name="cover" content="id"/> (EPUB 2). hrefs are
    relative to the OPF folder and URL-encoded (`cover%20art.png`)."""
    raise NotImplementedError


def _fb2_metadata(path: Path) -> Metadata:
    """book-title, author (first, middle, last joined by spaces) and lang from title-info."""
    raise NotImplementedError


def _cbz_cover(z: zipfile.ZipFile) -> bytes:
    """The first image in natural order (page2 before page10)."""
    raise NotImplementedError


def _pdf_inspect(path: Path) -> tuple[Metadata, bytes | None]:
    """Open with pymupdf (filetype="pdf"). Reject `needs_pass` PDFs and PDFs with
    no pages. Return metadata and page 1 rendered to PNG with its longer side
    about 600 px. A cover that fails to render is None, not a failed import."""
    raise NotImplementedError


def _image_ext(data: bytes) -> str | None:
    """'.png', '.jpg', '.gif' or '.webp' from the image signature, else None."""
    raise NotImplementedError

## Import

In [ ]:
def import_book(
    src: Path,
    library_dir: Path,
    original_name: str | None = None,
    max_bytes: int = MAX_BYTES,
) -> BookRecord:
    """Validate, inspect and store a book.

    Order: reject empty and oversized files, detect the format, inspect it
    (DRM, metadata, cover), and only then hash and copy it to
    `library_dir/<sha256>.<format>`, plus `<sha256>.cover<ext>` if there is a
    cover. Parser errors from detection or inspection (BadZipFile, ParseError,
    struct.error, KeyError, IndexError, AttributeError, UnicodeDecodeError,
    zlib.error, RuntimeError, EOFError) become ImportRejected("corrupt file"). The title falls back to the stem of
    `original_name` (or of `src`).
    """
    raise NotImplementedError

## Test fixtures

Real books are large and copyrighted, so the tests build tiny ones.

In [ ]:
FAKE_PNG = b"\x89PNG\r\n\x1a\n" + b"fake-image-bytes"


def make_pdf(path, title="Paper Title", author="P. Author", user_pw=None, owner_pw=None):
    doc = pymupdf.open()
    doc.new_page().insert_text((72, 72), "Hello, Hudhud.")
    doc.set_metadata({"title": title, "author": author})
    if owner_pw:
        doc.save(path, encryption=pymupdf.PDF_ENCRYPT_AES_256,
                 owner_pw=owner_pw, user_pw=user_pw or "")
    else:
        doc.save(path)
    doc.close()


def make_epub(path, title="کتاب نمونه", author="نویسنده", language="fa",
              cover="epub3", encryption_alg=None, rights=False):
    cover_meta = '<meta name="cover" content="cov"/>' if cover == "epub2" else ""
    cover_props = ' properties="cover-image"' if cover == "epub3" else ""
    opf = f"""<?xml version="1.0" encoding="UTF-8"?>
<package xmlns="http://www.idpf.org/2007/opf" version="3.0" unique-identifier="id">
  <metadata xmlns:dc="http://purl.org/dc/elements/1.1/">
    <dc:identifier id="id">urn:uuid:1</dc:identifier>
    <dc:title>{title}</dc:title>
    <dc:creator>{author}</dc:creator>
    <dc:language>{language}</dc:language>
    {cover_meta}
  </metadata>
  <manifest>
    <item id="cov" href="images/cover%20art.png" media-type="image/png"{cover_props}/>
    <item id="c1" href="text/c1.xhtml" media-type="application/xhtml+xml"/>
  </manifest>
  <spine><itemref idref="c1"/></spine>
</package>"""
    container = """<?xml version="1.0"?>
<container version="1.0" xmlns="urn:oasis:names:tc:opendocument:xmlns:container">
  <rootfiles>
    <rootfile full-path="OEBPS/content.opf" media-type="application/oebps-package+xml"/>
  </rootfiles>
</container>"""
    with zipfile.ZipFile(path, "w") as z:
        z.writestr("mimetype", "application/epub+zip", compress_type=zipfile.ZIP_STORED)
        z.writestr("META-INF/container.xml", container)
        z.writestr("OEBPS/content.opf", opf)
        z.writestr("OEBPS/text/c1.xhtml", "<html><body><p>Salam</p></body></html>")
        z.writestr("OEBPS/images/cover art.png", FAKE_PNG)
        if encryption_alg:
            z.writestr("META-INF/encryption.xml",
                '<encryption xmlns="urn:oasis:names:tc:opendocument:xmlns:container" '
                'xmlns:enc="http://www.w3.org/2001/04/xmlenc#"><enc:EncryptedData>'
                f'<enc:EncryptionMethod Algorithm="{encryption_alg}"/>'
                "</enc:EncryptedData></encryption>")
        if rights:
            z.writestr("META-INF/rights.xml", "<rights/>")


def make_mobi(path, title="Test Book", author="A. Writer", language="en",
              version=6, encryption=0):
    exth_records = b""
    fields = [(100, author), (503, title), (524, language)]
    for rtype, value in fields:
        data = value.encode("utf-8")
        exth_records += struct.pack(">II", rtype, 8 + len(data)) + data
    exth = b"EXTH" + struct.pack(">II", 12 + len(exth_records), len(fields)) + exth_records
    header_len = 232
    full_name = title.encode("utf-8")
    rec0 = bytearray(16 + header_len)
    struct.pack_into(">H", rec0, 12, encryption)
    rec0[16:20] = b"MOBI"
    struct.pack_into(">I", rec0, 20, header_len)
    struct.pack_into(">I", rec0, 28, 65001)
    struct.pack_into(">I", rec0, 36, version)
    struct.pack_into(">II", rec0, 84, 16 + header_len + len(exth), len(full_name))
    struct.pack_into(">I", rec0, 128, 0x40)
    rec0 += exth + full_name + b"\0\0"
    pdb = bytearray(78)
    pdb[0:4] = b"test"
    pdb[60:68] = b"BOOKMOBI"
    struct.pack_into(">H", pdb, 76, 1)
    pdb += struct.pack(">I", 78 + 8 + 2) + b"\0\0\0\0" + b"\0\0"
    Path(path).write_bytes(bytes(pdb) + bytes(rec0))


FB2_SAMPLE = """<?xml version="1.0" encoding="utf-8"?>
<FictionBook xmlns="http://www.gribuser.ru/xml/fictionbook/2.0">
  <description><title-info>
    <author><first-name>Lev</first-name><last-name>Tolstoy</last-name></author>
    <book-title>War and Peace</book-title><lang>ru</lang>
  </title-info></description>
  <body><section><p>Well, Prince...</p></section></body>
</FictionBook>"""


def corrupt_member(path, name):
    """Overwrite the compressed bytes of one entry, keeping the zip structure."""
    data = bytearray(Path(path).read_bytes())
    header = data.find(name.encode()) - 30  # first hit is the local header
    csize, = struct.unpack_from("<I", data, header + 18)
    name_len, extra_len = struct.unpack_from("<HH", data, header + 26)
    start = header + 30 + name_len + extra_len
    data[start:start + csize] = b"\xff" * csize
    Path(path).write_bytes(bytes(data))


def encrypt_flag(path):
    """Mark every entry as password-protected without encrypting it."""
    data = bytearray(Path(path).read_bytes())
    for sig, flag_offset in ((b"PK\x03\x04", 6), (b"PK\x01\x02", 8)):
        i = data.find(sig)
        while i != -1:
            data[i + flag_offset] |= 1
            i = data.find(sig, i + 4)
    Path(path).write_bytes(bytes(data))


def make_cbz(path):
    with zipfile.ZipFile(path, "w") as z:
        z.writestr("page10.png", b"\x89PNG\r\n\x1a\nten")
        z.writestr("page2.png", b"\x89PNG\r\n\x1a\ntwo")
        z.writestr("page1.png", b"\x89PNG\r\n\x1a\none")

## Tests: identity and format

In [ ]:
%%ipytest -q
import pytest


def test_sha256_matches_hashlib(tmp_path):
    p = tmp_path / "x.bin"
    p.write_bytes(b"hudhud" * 1000)
    assert sha256_file(p) == hashlib.sha256(b"hudhud" * 1000).hexdigest()


def test_detect_every_format(tmp_path):
    make_pdf(tmp_path / "a.pdf")
    make_epub(tmp_path / "a.epub")
    make_mobi(tmp_path / "a.mobi")
    make_mobi(tmp_path / "a.azw3", version=8)
    make_cbz(tmp_path / "a.cbz")
    (tmp_path / "a.fb2").write_text(FB2_SAMPLE, encoding="utf-8")
    (tmp_path / "a.txt").write_text("سلام دنیا\nHello world\n", encoding="utf-8")
    found = {p.suffix[1:]: detect_format(p) for p in tmp_path.iterdir()}
    assert found == {"pdf": "pdf", "epub": "epub", "mobi": "mobi", "azw3": "azw3",
                     "cbz": "cbz", "fb2": "fb2", "txt": "txt"}


def test_extension_is_ignored(tmp_path):
    p = tmp_path / "really-a-pdf.epub"
    make_pdf(p)
    assert detect_format(p) == "pdf"


def test_binary_garbage_is_unsupported(tmp_path):
    p = tmp_path / "x.bin"
    p.write_bytes(bytes(range(256)) * 4)
    with pytest.raises(ImportRejected) as e:
        detect_format(p)
    assert e.value.reason == "unsupported format"


def test_broken_zip_is_corrupt(tmp_path):
    p = tmp_path / "x.epub"
    p.write_bytes(b"PK\x03\x04" + b"not really a zip" * 10)
    with pytest.raises(ImportRejected) as e:
        detect_format(p)
    assert e.value.reason == "corrupt file"

## Tests: import

In [ ]:
%%ipytest -q
import pytest


def test_import_pdf_metadata_and_cover(tmp_path):
    src = tmp_path / "in.pdf"
    make_pdf(src, title="Thinking, Fast and Slow", author="Daniel Kahneman")
    rec = import_book(src, tmp_path / "lib")
    assert rec.id == sha256_file(src)
    assert (rec.title, rec.author, rec.format) == ("Thinking, Fast and Slow", "Daniel Kahneman", "pdf")
    assert Path(rec.file_path) == tmp_path / "lib" / f"{rec.id}.pdf"
    assert Path(rec.cover_path).read_bytes().startswith(b"\x89PNG")
    assert rec.file_size == src.stat().st_size


def test_import_is_idempotent(tmp_path):
    src = tmp_path / "in.pdf"
    make_pdf(src)
    a = import_book(src, tmp_path / "lib")
    b = import_book(src, tmp_path / "lib")
    assert a == b
    assert sorted(p.name for p in (tmp_path / "lib").iterdir()) == sorted(
        [f"{a.id}.pdf", f"{a.id}.cover.png"])


def test_password_pdf_rejected(tmp_path):
    src = tmp_path / "locked.pdf"
    make_pdf(src, user_pw="user", owner_pw="owner")
    with pytest.raises(ImportRejected):
        import_book(src, tmp_path / "lib")


def test_owner_password_only_pdf_imports(tmp_path):
    src = tmp_path / "restricted.pdf"
    make_pdf(src, title="Restricted", owner_pw="owner")
    assert import_book(src, tmp_path / "lib").title == "Restricted"


def test_epub3_farsi_metadata_and_cover(tmp_path):
    src = tmp_path / "in.epub"
    make_epub(src)
    rec = import_book(src, tmp_path / "lib")
    assert (rec.title, rec.author, rec.language, rec.format) == ("کتاب نمونه", "نویسنده", "fa", "epub")
    assert Path(rec.cover_path).read_bytes() == FAKE_PNG


def test_epub2_cover_meta(tmp_path):
    src = tmp_path / "in.epub"
    make_epub(src, cover="epub2")
    assert Path(import_book(src, tmp_path / "lib").cover_path).read_bytes() == FAKE_PNG


def test_epub_without_cover(tmp_path):
    src = tmp_path / "in.epub"
    make_epub(src, cover=None)
    assert import_book(src, tmp_path / "lib").cover_path is None


def test_epub_font_obfuscation_is_not_drm(tmp_path):
    src = tmp_path / "in.epub"
    make_epub(src, encryption_alg="http://www.idpf.org/2008/embedding")
    assert import_book(src, tmp_path / "lib").format == "epub"


@pytest.mark.parametrize("kwargs", [
    {"encryption_alg": "http://www.w3.org/2001/04/xmlenc#aes128-cbc"},
    {"rights": True},
])
def test_epub_drm_rejected(tmp_path, kwargs):
    src = tmp_path / "in.epub"
    make_epub(src, **kwargs)
    with pytest.raises(ImportRejected):
        import_book(src, tmp_path / "lib")


def test_mobi_and_azw3_metadata(tmp_path):
    make_mobi(tmp_path / "a.mobi", title="Shahnameh", author="Ferdowsi", language="fa")
    make_mobi(tmp_path / "b.azw3", title="Gulistan", author="Saadi", version=8)
    a = import_book(tmp_path / "a.mobi", tmp_path / "lib")
    b = import_book(tmp_path / "b.azw3", tmp_path / "lib")
    assert (a.title, a.author, a.language, a.format) == ("Shahnameh", "Ferdowsi", "fa", "mobi")
    assert (b.title, b.format, b.cover_path) == ("Gulistan", "azw3", None)


def test_mobi_drm_rejected(tmp_path):
    make_mobi(tmp_path / "a.azw3", version=8, encryption=2)
    with pytest.raises(ImportRejected):
        import_book(tmp_path / "a.azw3", tmp_path / "lib")


def test_fb2_metadata(tmp_path):
    src = tmp_path / "in.fb2"
    src.write_text(FB2_SAMPLE, encoding="utf-8")
    rec = import_book(src, tmp_path / "lib")
    assert (rec.title, rec.author, rec.language) == ("War and Peace", "Lev Tolstoy", "ru")


def test_cbz_cover_is_first_page_in_natural_order(tmp_path):
    src = tmp_path / "comic.cbz"
    make_cbz(src)
    rec = import_book(src, tmp_path / "lib", original_name="My Comic.cbz")
    assert rec.title == "My Comic"
    assert Path(rec.cover_path).read_bytes().endswith(b"one")


def test_txt_title_from_original_name(tmp_path):
    src = tmp_path / "upload-123.tmp"
    src.write_text("یک روز...", encoding="utf-8")
    rec = import_book(src, tmp_path / "lib", original_name="Ruzi Ruzegari.txt")
    assert (rec.title, rec.format, rec.cover_path) == ("Ruzi Ruzegari", "txt", None)


def test_rejections_store_nothing(tmp_path):
    lib = tmp_path / "lib"
    empty = tmp_path / "empty.txt"
    empty.write_bytes(b"")
    big = tmp_path / "big.txt"
    big.write_text("x" * 100)
    locked = tmp_path / "locked.pdf"
    make_pdf(locked, user_pw="u", owner_pw="o")
    for src, kwargs in [(empty, {}), (big, {"max_bytes": 10}), (locked, {})]:
        with pytest.raises(ImportRejected):
            import_book(src, lib, **kwargs)
    assert not lib.exists() or list(lib.iterdir()) == []


def test_truncated_mobi_is_corrupt(tmp_path):
    src = tmp_path / "a.mobi"
    make_mobi(src)
    src.write_bytes(src.read_bytes()[:100])
    with pytest.raises(ImportRejected) as e:
        import_book(src, tmp_path / "lib")
    assert e.value.reason == "corrupt file"

## Tests: hostile and broken files

In [ ]:
%%ipytest -q
import pytest


def test_huge_archive_entry_rejected(tmp_path):
    src = tmp_path / "bomb.cbz"
    with zipfile.ZipFile(src, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.writestr("page1.png", b"\x89PNG\r\n\x1a\n" + bytes(MAX_MEMBER_BYTES + 1))
    with pytest.raises(ImportRejected) as e:
        import_book(src, tmp_path / "lib")
    assert e.value.reason == "archive entry too large"


def test_bad_compressed_data_is_corrupt(tmp_path):
    src = tmp_path / "in.epub"
    with zipfile.ZipFile(src, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.writestr("mimetype", "application/epub+zip" * 20)
    corrupt_member(src, "mimetype")
    with pytest.raises(ImportRejected) as e:
        import_book(src, tmp_path / "lib")
    assert e.value.reason == "corrupt file"


def test_password_protected_comic_page_is_corrupt(tmp_path):
    src = tmp_path / "in.cbz"
    make_cbz(src)
    encrypt_flag(src)
    with pytest.raises(ImportRejected) as e:
        import_book(src, tmp_path / "lib")
    assert e.value.reason == "corrupt file"


def test_extreme_page_shape_still_imports(tmp_path):
    src = tmp_path / "tall.pdf"
    doc = pymupdf.open()
    doc.new_page(width=3, height=14400)
    doc.save(src)
    doc.close()
    assert import_book(src, tmp_path / "lib").format == "pdf"


def test_epub_without_mimetype_is_still_checked_for_drm(tmp_path):
    src = tmp_path / "in.epub"
    make_epub(src, rights=True)
    with zipfile.ZipFile(src) as z:
        entries = {n: z.read(n) for n in z.namelist() if n != "mimetype"}
    with zipfile.ZipFile(src, "w") as z:
        for name, data in entries.items():
            z.writestr(name, data)
    with pytest.raises(ImportRejected) as e:
        import_book(src, tmp_path / "lib")
    assert "DRM" in e.value.reason


def test_office_document_is_not_a_comic(tmp_path):
    src = tmp_path / "report.docx"
    with zipfile.ZipFile(src, "w") as z:
        z.writestr("word/document.xml", "<w:document/>")
        z.writestr("word/media/image1.png", FAKE_PNG)
    with pytest.raises(ImportRejected) as e:
        detect_format(src)
    assert e.value.reason == "unsupported format"

## Graduate

When everything is green:

1. Create `hub/src/hudhud_hub/library.py`. Start it with the SPDX line, then
   copy the imports (minus `ipytest`), the constants and types cell, and your
   function cells.
2. Create `hub/tests/test_library.py`. Copy the fixture cell and both test
   cells without the `%%ipytest` lines, and add
   `from hudhud_hub.library import *` at the top.
3. `uv run --project hub pytest hub/tests/test_library.py -q`